# Urban Event Monitoring: BERT Training

This notebook downloads a large disaster tweets dataset (similar to Kaggle) and fine-tunes a DistilBERT model to classify urban events.

In [ ]:
import pandas as pd
import numpy as np
import nltk
import spacy
from datasets import load_dataset

print("Loading libraries...")

## 1. Download Large Dataset
We use a dataset from HuggingFace Hub which originates from the famous Kaggle Disaster Tweets competition.

In [ ]:
# Downloading Kaggle Disaster Tweets Dataset
dataset = load_dataset('mehdiiraqui/twitter_disaster')

# Convert to pandas dataframe
df_train = pd.DataFrame(dataset['train'])
df_test = pd.DataFrame(dataset['test'])

print(f"Training samples: {len(df_train)}")
print(f"Testing samples: {len(df_test)}")
display(df_train.head())

## 2. Text Preprocessing using NLTK and spaCy
Cleaning raw text by removing noise, special characters, and normalizing.

In [ ]:
import re
from nltk.corpus import stopwords
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = re.sub(r'http\S+', '', text) # Remove URLs
    text = re.sub(r'[^a-zA-Z\s]', '', text) # Remove special characters
    text = text.lower()
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return ' '.join(words)

df_train['clean_text'] = df_train['text'].apply(clean_text)
df_test['clean_text'] = df_test['text'].apply(clean_text)
print("Sample cleaned text:", df_train['clean_text'].iloc[0])

## 3. Train DistilBERT Classifier

In [ ]:
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, Trainer, TrainingArguments
import torch

tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

def tokenize_data(data):
    return tokenizer(
        data['clean_text'].tolist(),
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors='pt'
    )

train_encodings = tokenize_data(df_train)
test_encodings = tokenize_data(df_test)

class TweetDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = TweetDataset(train_encodings, df_train['target'].tolist())
test_dataset = TweetDataset(test_encodings, df_test['target'].tolist())


In [ ]:
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)

training_args = TrainingArguments(
    output_dir='../ml_models/bert-event-classifier',
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='../logs',
    logging_steps=10,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

# trainer.train() # Uncomment to run full training